In [ ]:
# Module 3 — RAG Evaluation
import json, faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Load the jobs
data_path = Path.cwd() / "data" / "job_postings.json"
if not data_path.exists():
    data_path = Path.cwd().parent / "data" / "job_postings.json"
with open(data_path, "r", encoding="utf-8") as f:
    jobs = json.load(f)

# Embed jobs + build the FAISS index
model = SentenceTransformer("all-MiniLM-L6-v2")
job_texts = [f"{j['title']}. {j['description']}" for j in jobs]
job_vectors = model.encode(job_texts, normalize_embeddings=True)
index = faiss.IndexFlatIP(job_vectors.shape[1])
index.add(job_vectors)

print(f"Ready: {index.ntotal} jobs indexed.")

In [ ]:
test_set = [
    {"query": "I build deep learning models for computer vision using CNNs and PyTorch",
     "correct_title": "Deep Learning Researcher"},
    {"query": "I design REST APIs and microservices in Python and Go with Docker",
     "correct_title": "Backend Software Engineer"},
    {"query": "I run social media campaigns and manage Instagram advertising budgets",
     "correct_title": "Digital Marketing Manager"},
    {"query": "I build financial models and analyze market trends in Excel and Python",
     "correct_title": "Financial Analyst"},
    {"query": "I design user interfaces and build prototypes in Figma",
     "correct_title": "Product Designer"},
    {"query": "I build ETL data pipelines with Spark and Airflow on AWS",
     "correct_title": "Data Engineer"},
]
print(f"{len(test_set)} labeled test queries ready.")

In [ ]:
def search_titles(query, k=len(jobs)):
    """Return job titles ranked by fit for a query (best first)."""
    q_vec = model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_vec, k)
    return [jobs[i]["title"] for i in indices[0]]

In [ ]:
def evaluate(test_set, k=3):
    hits = 0
    reciprocal_ranks = []

    for item in test_set:
        ranked = search_titles(item["query"])
        correct = item["correct_title"]

        # Recall@k: is the correct job in the top k?
        if correct in ranked[:k]:
            hits += 1

        # Reciprocal rank: 1 / position of the correct job (0 if missing)
        if correct in ranked:
            position = ranked.index(correct) + 1
            reciprocal_ranks.append(1 / position)
        else:
            reciprocal_ranks.append(0)

    recall_at_k = hits / len(test_set)
    mrr = sum(reciprocal_ranks) / len(test_set)
    return recall_at_k, mrr

recall, mrr = evaluate(test_set, k=3)
print(f"Recall@3: {recall:.2f}")
print(f"MRR:      {mrr:.2f}")

In [ ]:
print(f"{'Correct job':32}{'Rank':>5}")
print("-" * 37)
for item in test_set:
    ranked = search_titles(item["query"])
    correct = item["correct_title"]
    rank = ranked.index(correct) + 1 if correct in ranked else "—"
    print(f"{correct:32}{str(rank):>5}")